# Huấn luyện mô hình cháy / khói cho SafeSite (YOLO11)

**Trước khi chạy** (cột bên phải của Kaggle):
1. *Settings → Accelerator* = **GPU T4** (hoặc P100). *Settings → Internet* = **On** (cần để cài `ultralytics` và tải trọng số `yolo11*.pt`).
2. *Add Input* → thêm một dataset **định dạng YOLO** về lửa/khói (ví dụ D-Fire). Tự kiểm tra: tên dataset, **giấy phép** (nhiều bộ chỉ cho phép phi thương mại) và **thứ tự lớp**.
3. Lần đầu đặt `EPOCHS = 3` ở ô dưới để chắc mọi thứ chạy, sau đó mới tăng lên (vài chục epoch).

**Kết quả:** file `fire_smoke_best.pt` trong tab *Output* → tải về, đặt vào `safety-monitor/models/`, rồi trong dashboard: *Cài đặt → Cháy / khói → Mô hình*.

In [ ]:
# ====== CẤU HÌNH – chỉnh ở đây ======
MODEL = "yolo11n.pt"     # n = nhẹ, chạy được trên CPU; s/m chính xác hơn nhưng chậm hơn khi chạy trong app
EPOCHS = 3               # chạy thử 3; huấn luyện thật: 50
IMGSZ = 640
BATCH = 32               # hết bộ nhớ GPU thì giảm xuống 16
DATASET_ROOT = None      # None = tự dò trong /kaggle/input; hoặc ghi đường dẫn thư mục gốc dataset
CLASS_NAMES = None       # None = đọc từ file .yaml/classes.txt của dataset; hoặc ghi tay, VD ["smoke", "fire"]

In [ ]:
!pip -q install -U ultralytics
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU – bật Accelerator!")

In [ ]:
import shutil
from collections import Counter
from pathlib import Path

import yaml


def norm_split(name: str) -> str:
    n = name.lower()
    return "train" if "train" in n else "val" if n.startswith("val") else "test" if "test" in n else n


def find_splits(root) -> dict:
    """Dò các thư mục ảnh YOLO. Hỗ trợ hai kiểu bố trí:
       A) <root>/<split>/images + <root>/<split>/labels      (D-Fire)
       B) <root>/images/<split> + <root>/labels/<split>"""
    found = {}
    for img_dir in sorted(Path(root).rglob("images")):
        lbl_dir = img_dir.parent / "labels"
        if not (img_dir.is_dir() and lbl_dir.is_dir()):
            continue
        subs = [d for d in img_dir.iterdir() if d.is_dir() and (lbl_dir / d.name).is_dir()]
        if subs:                                              # kiểu B
            for d in subs:
                found.setdefault(norm_split(d.name), d)
        else:                                                 # kiểu A
            found.setdefault(norm_split(img_dir.parent.name), img_dir)
    return found


def read_class_names(root):
    for y in sorted(Path(root).rglob("*.y*ml")):
        try:
            names = yaml.safe_load(y.read_text()).get("names")
        except Exception:
            continue
        if isinstance(names, dict):
            return [names[k] for k in sorted(names)]
        if isinstance(names, list):
            return names
    for t in sorted(Path(root).rglob("classes.txt")):
        return [l.strip() for l in t.read_text().splitlines() if l.strip()]
    return None


def label_dir_of(img_dir: Path) -> Path:
    return Path(str(img_dir).replace("/images", "/labels"))


def class_histogram(img_dir: Path) -> Counter:
    c = Counter()
    for f in label_dir_of(img_dir).glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                c[int(line.split()[0])] += 1
    return c

In [ ]:
root = Path(DATASET_ROOT) if DATASET_ROOT else Path("/kaggle/input")
splits = find_splits(root)
print("Tìm thấy:", {k: str(v) for k, v in splits.items()})
assert "train" in splits, f"Không thấy thư mục train trong {root}. Kiểm tra đã Add Input chưa, hoặc đặt DATASET_ROOT."
val_key = "val" if "val" in splits else "test" if "test" in splits else None
assert val_key, "Dataset không có tập val/test"
if val_key == "test":
    print("⚠ Dataset không có tập val, dùng tập test để theo dõi (đánh giá cuối nên dùng dữ liệu riêng).")

names = CLASS_NAMES or read_class_names(root)
hist = class_histogram(splits["train"])
print("Số nhãn theo id lớp (train):", dict(sorted(hist.items())))
if not names:
    names = ["smoke", "fire"]
    print("⚠ Không đọc được tên lớp từ dataset → GIẢ ĐỊNH", names, "(đúng với D-Fire). KIỂM TRA LẠI bằng tài liệu dataset!")
assert max(hist) < len(names), f"Nhãn có id lớp {max(hist)} nhưng chỉ có {len(names)} tên lớp {names}"
print("Tên lớp:", names)
assert any(("fire" in n.lower() or "flame" in n.lower()) for n in names) or any("smoke" in n.lower() for n in names), \
    "Không lớp nào có chữ fire/flame/smoke → app sẽ bỏ qua mô hình này. Đổi tên lớp (CLASS_NAMES)."

DATA_YAML = Path("/kaggle/working/fire_smoke.yaml")
DATA_YAML.write_text(yaml.safe_dump({"path": "", "train": str(splits["train"]), "val": str(splits[val_key]),
                                     "names": {i: n for i, n in enumerate(names)}}, allow_unicode=True))
print(DATA_YAML.read_text())

### Huấn luyện
Cảnh báo kiểu *"labels cache not writeable"* là bình thường vì `/kaggle/input` chỉ đọc. Nếu phiên bị ngắt giữa chừng: chạy lại ô cài đặt, rồi `YOLO("/kaggle/working/runs/fire_smoke/weights/last.pt").train(resume=True)`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0, workers=2,
            patience=15, amp=True, plots=True, save_period=5,
            project="/kaggle/working/runs", name="fire_smoke", exist_ok=True)

In [ ]:
best = Path(model.trainer.best)
m = YOLO(str(best)).val(data=str(DATA_YAML), imgsz=IMGSZ, device=0)
print(f"mAP50 = {m.box.map50:.3f} | mAP50-95 = {m.box.map:.3f} | P = {m.box.mp:.3f} | R = {m.box.mr:.3f}")
for i, n in m.names.items():
    p, r, ap50, ap = m.box.class_result(i)
    print(f"  {n:8s} P={p:.3f} R={r:.3f} AP50={ap50:.3f} AP50-95={ap:.3f}")

In [ ]:
out = Path("/kaggle/working/fire_smoke_best.pt")
shutil.copy(best, out)
print("Đã lưu:", out, f"({out.stat().st_size / 1e6:.1f} MB) – tải ở tab Output")
print("Tên lớp trong mô hình:", YOLO(str(out)).names)

# Xem thử vài ảnh val
import random
from IPython.display import Image, display
imgs = sorted(splits[val_key].glob("*.jpg")) + sorted(splits[val_key].glob("*.png"))
sample = random.sample(imgs, min(4, len(imgs)))
res = YOLO(str(out)).predict(sample, conf=0.4, save=True, project="/kaggle/working/runs", name="preview", exist_ok=True, verbose=False)
for r in res:
    display(Image(filename=str(Path(r.save_dir) / Path(r.path).name)))

### Dùng trong SafeSite
1. Tải `fire_smoke_best.pt` (tab *Output* bên phải) → đặt vào `safety-monitor/models/`.
2. Dashboard → **Cài đặt → Cháy / khói → Mô hình** = `models/fire_smoke_best.pt` → *Lưu cài đặt* → chạy lại camera.
3. Thử trên vài video lửa/khói thật và đo báo nhầm/giờ bằng `tools/evaluate.py` trước khi tin vào kết quả. Mô hình học từ ảnh tĩnh nên có thể nhầm đèn, nắng chiếu, sương mù – chỉnh `conf` và `min_hits` ở Cài đặt cho phù hợp.